# Edge ET0 Forecasting — From-Scratch Reproduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SarvanshRaj/research-repro-luo2025-et0/blob/main/notebooks/research_colab.ipynb)

Reproduction of Luo et al. (2025) — Benchmarking deep learning models for ET0 forecasting on edge devices.

**Primary result:** GRU achieves R² = 0.966 ± 0.001 (n=3 seeds)

In [ ]:
# Install dependencies
!pip install torch numpy pandas scikit-learn matplotlib tqdm -q

In [ ]:
# Clone repo
!git clone https://github.com/SarvanshRaj/research-repro-luo2025-et0.git
%cd research-repro-luo2025-et0

In [ ]:
# Quick smoke test — 1 epoch
from src.train import train_model
from src.evaluate import evaluate_model, print_metrics
from src.utils import get_device
import torch

model, history, data, test_loader = train_model(
    model_name='gru', lookback=12, epochs=1, batch_size=64,
    lr=5e-4, seed=42, save_dir='runs/colab_smoke', n_samples=1000, patience=100
)

device = get_device()
ckpt = torch.load('runs/colab_smoke/best_gru_seed42.pt', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
results = evaluate_model(model, test_loader, device, scaler_y=data['scaler_y'])
print_metrics(results, 'GRU (smoke)')
print('Smoke test PASSED')

In [ ]:
# Full training — GRU, 100 epochs
model, history, data, test_loader = train_model(
    model_name='gru', lookback=12, epochs=100, batch_size=64,
    lr=5e-4, seed=42, save_dir='runs/colab_full', n_samples=8760, patience=25
)

device = get_device()
ckpt = torch.load('runs/colab_full/best_gru_seed42.pt', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
results = evaluate_model(model, test_loader, device, scaler_y=data['scaler_y'])
print_metrics(results, 'GRU (full)')

In [ ]:
# Visualize predictions
from src.utils import plot_predictions

plot_predictions(
    results['y_true'], results['y_pred'],
    title='GRU ET0 Forecasting',
    save_path='experiments/figures/gru_predictions.png'
)
from IPython.display import Image
Image('experiments/figures/gru_predictions.png')

In [ ]:
# Multi-seed validation
from src.evaluate import compute_metrics
import numpy as np

seeds = [42, 123, 2026]
r2_scores = []

for seed in seeds:
    model_s, hist_s, data_s, test_s = train_model(
        model_name='gru', lookback=12, epochs=100, batch_size=64,
        lr=5e-4, seed=seed, save_dir=f'runs/colab_seed{seed}',
        n_samples=8760, patience=25
    )
    ckpt_s = torch.load(f'runs/colab_seed{seed}/best_gru_seed{seed}.pt', map_location=device)
    model_s.load_state_dict(ckpt_s['model_state_dict'])
    res_s = evaluate_model(model_s, test_s, device, scaler_y=data_s['scaler_y'])
    r2_scores.append(res_s['metrics']['r2'])
    print(f'Seed {seed}: R² = {res_s["metrics"]["r2"]:.4f}')

print(f'\nR² = {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f} (n={len(seeds)})')
print(f'Paper target: R² ≥ 0.984 ± 0.005')
print(f'Reproduction within ±5%: {abs(np.mean(r2_scores) - 0.9888) / 0.9888 * 100:.1f}%')